# 01 - Extracción de datos
## Objetivo
Extraer archivos Parquet desde raw, construir inventario técnico y detectar corruptos.

In [1]:
import os, sys, json
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd()))
if os.path.basename(PROJECT_ROOT) == "notebooks":
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from src.utils import load_config, create_spark_session, generate_process_id, ensure_dir, setup_logger

config = load_config(os.path.join(PROJECT_ROOT, "config", "etl_config.yaml"))
sys.path.insert(0, os.path.abspath(".."))

from src.utils import load_config, create_spark_session, generate_process_id, ensure_dir, setup_logger

config = load_config("config/etl_config.yaml")
process_id = generate_process_id()
log = setup_logger("extraction")

print(f"Process ID: {process_id}")
spark = create_spark_session(config)

# --- Extracción ---
from src.extract import run_extraction

inventory = run_extraction(spark, config, process_id, log)

print(f"\n{'='*60}")
print(f"Total archivos procesados: {len(inventory)}")
print(f"{'='*60}")

ok = [i for i in inventory if i["read_status"] == "SUCCESS"]
failed = [i for i in inventory if i["read_status"] == "FAILED"]
print(f"  OK: {len(ok)}")
print(f"  FAILED: {len(failed)}")

print(f"\n{'='*60}")
print("INVENTARIO TÉCNICO")
print(f"{'='*60}")
for i in inventory:
    status = "OK" if i["read_status"] == "SUCCESS" else "FAIL"
    print(f"  {i['file_name']:45s} | {status:4s} | registros={i['record_count']:>10d} | cols={i['column_count']} | {i.get('recovery_category','')}")

if failed:
    print(f"\n{'='*60}")
    print("ARCHIVOS FALLIDOS")
    print(f"{'='*60}")
    for f in failed:
        print(f"  {f['file_name']:45s} | {f['error_message'][:100]}")

# Guardar inventario como JSON
audit_dir = config["paths"]["audit"]
ensure_dir(audit_dir)
with open(f"{audit_dir}/audit_file_inventory_{process_id}.json", "w") as f:
    json.dump(inventory, f, indent=2, default=str)

print(f"\nInventario guardado en {audit_dir}/audit_file_inventory_{process_id}.json")
spark.stop()

Process ID: ETL_20260613_143829_11d6e271


2026-06-13 09:38:35,230 | INFO     | extraction | Extracting inventory for service: yellow
2026-06-13 09:38:39,393 | INFO     | extraction | Extracting inventory for service: green
2026-06-13 09:38:39,633 | INFO     | extraction | Extracting inventory for service: fhvhv
2026-06-13 09:38:39,753 | INFO     | extraction | Extracting bad_parquet files
2026-06-13 09:38:39,856 | WARNING  | extraction | bad_parquet failed: ARROW-GH-41317.parquet: [PARQUET_TYPE_ILLEGAL] Illegal Parquet type: INT32 (TIME(MILLIS,true)). SQLSTATE: 42846
2026-06-13 09:38:39,918 | WARNING  | extraction | bad_parquet failed: ARROW-GH-41321.parquet: [PARQUET_TYPE_ILLEGAL] Illegal Parquet type: INT32 (TIME(MILLIS,true)). SQLSTATE: 42846



Total archivos procesados: 16
  OK: 14
  FAILED: 2

INVENTARIO TÉCNICO
  yellow_tripdata_2020-01.parquet               | OK   | registros=   6405008 | cols=19 | RECOVERABLE
  yellow_tripdata_2021-01.parquet               | OK   | registros=   1369769 | cols=19 | RECOVERABLE
  yellow_tripdata_2022-01.parquet               | OK   | registros=   2463931 | cols=19 | RECOVERABLE
  yellow_tripdata_2022-02.parquet               | OK   | registros=   2979431 | cols=19 | RECOVERABLE
  yellow_tripdata_2023-01.parquet               | OK   | registros=   3066766 | cols=19 | RECOVERABLE
  yellow_tripdata_2023-02.parquet               | OK   | registros=   2913955 | cols=19 | RECOVERABLE
  yellow_tripdata_2023-03.parquet               | OK   | registros=   3403766 | cols=19 | RECOVERABLE
  yellow_tripdata_2023-04.parquet               | OK   | registros=   3288250 | cols=19 | RECOVERABLE
  green_tripdata_2023-01.parquet                | OK   | registros=     68211 | cols=20 | RECOVERABLE
  green_tr